---
## 0. Imports

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from pyproj import Transformer
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import pandas as pd
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10

print("Imports OK.")

---
## 1. Configuration

Define the mask pairs and comparison ROI.

In [ ]:
PAIRS = [
    {
        "label": "2024-01-15",
        "cbers4": "../output/cbers4_multi/water_mask_scene_20240115.tif",
        "s2_scl": "../output/sentinel2_multi/s2_scl_water_20240115.tif",
    },
    {
        "label": "2024-02-08",
        "cbers4": "../output/cbers4_multi/water_mask_scene_20240208.tif",
        "s2_scl": "../output/sentinel2_multi/s2_scl_water_20240208.tif",
    },
    # --- ADD MORE PAIRS HERE ---
    # {
    #     "label": "YYYY-MM-DD",
    #     "cbers4": "../output/cbers4_multi/water_mask_scene_YYYYMMDD.tif",
    #     "s2_scl": "../output/sentinel2_multi/s2_scl_water_YYYYMMDD.tif",
    # },
]

# Comparison ROI [west, south, east, north] in EPSG:4326
COMPARE_BBOX = [-46.0, -23.4, -45.7, -23.1]

TARGET_RES = 64  # meters (CBERS-4 WFI resolution)

OUTPUT_DIR = "../output/comparison_multi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Pairs to compare: {len(PAIRS)}")
for p in PAIRS:
    c_ok = os.path.exists(p['cbers4'])
    s_ok = os.path.exists(p['s2_scl'])
    status = 'OK' if (c_ok and s_ok) else f"MISSING: {'cbers4' if not c_ok else ''} {'s2_scl' if not s_ok else ''}"
    print(f"  {p['label']:12s}  {status}")

west, south, east, north = COMPARE_BBOX
width_km = (east - west) * 111 * np.cos(np.radians((south + north) / 2))
height_km = (north - south) * 111
print(f"\nComparison ROI: {COMPARE_BBOX} (~{width_km:.0f}×{height_km:.0f} km)")

---
## 2. Clip and Reproject Helper

In [ ]:
def clip_and_reproject(src_path, bbox_4326, target_crs, target_res):
    """Clip a raster to BBOX and reproject to target CRS/resolution."""
    west, south, east, north = bbox_4326
    tr = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    t_left, t_bottom = tr.transform(west, south)
    t_right, t_top = tr.transform(east, north)
    out_width = int(np.ceil((t_right - t_left) / target_res))
    out_height = int(np.ceil((t_top - t_bottom) / target_res))
    out_transform = transform_from_bounds(t_left, t_bottom, t_right, t_top, out_width, out_height)
    dst_data = np.zeros((out_height, out_width), dtype=np.uint8)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1), destination=dst_data,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=out_transform, dst_crs=target_crs,
            dst_nodata=255, resampling=Resampling.nearest,
        )
    return dst_data


def compute_metrics(test, reference, valid):
    """Compute accuracy metrics. Returns dict."""
    t = (test == 1) & valid
    r = (reference == 1) & valid
    tp = (t & r).sum()
    fp = (t & ~r).sum()
    fn = (~t & r).sum()
    tn = (~t & ~r & valid).sum()
    total = tp + fp + fn + tn
    oa = (tp + tn) / total if total > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    pe = ((tp+fp)*(tp+fn) + (fn+tn)*(fp+tn)) / (total**2) if total > 0 else 0
    kappa = (oa - pe) / (1 - pe) if (1 - pe) > 0 else 0
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
            "OA": oa, "Kappa": kappa, "Precision": prec, "Recall": rec, "F1": f1}

print("Helpers defined.")

---
## 3. Get Target CRS from First CBERS-4 File

In [ ]:
# Use the CRS from the first valid CBERS-4 file
first_valid = next(p for p in PAIRS if os.path.exists(p['cbers4']))
with rasterio.open(first_valid['cbers4']) as src:
    TARGET_CRS = src.crs
print(f"Target CRS: {TARGET_CRS}")
print(f"Target resolution: {TARGET_RES} m")

---
## 4. Process All Pairs

In [ ]:
all_metrics = []  # Collect metrics for summary table
pair_data = []    # Store arrays for plotting

for pair in PAIRS:
    label = pair["label"]
    print(f"\n{'='*50}")
    print(f"  Pair: {label}")
    print(f"{'='*50}")

    # Check files exist
    if not os.path.exists(pair['cbers4']):
        print(f"  SKIP — CBERS-4 file not found: {pair['cbers4']}")
        continue
    if not os.path.exists(pair['s2_scl']):
        print(f"  SKIP — Sentinel-2 file not found: {pair['s2_scl']}")
        continue

    # Clip both to common grid
    cbers_clip = clip_and_reproject(pair['cbers4'], COMPARE_BBOX, TARGET_CRS, TARGET_RES)
    scl_clip = clip_and_reproject(pair['s2_scl'], COMPARE_BBOX, TARGET_CRS, TARGET_RES)

    # Align shapes
    min_h = min(cbers_clip.shape[0], scl_clip.shape[0])
    min_w = min(cbers_clip.shape[1], scl_clip.shape[1])
    cbers_clip = cbers_clip[:min_h, :min_w]
    scl_clip = scl_clip[:min_h, :min_w]

    # Binarize
    cbers_water = np.where(cbers_clip == 255, 255,
                           np.where((cbers_clip >= 1) & (cbers_clip <= 4), 1, 0)).astype(np.uint8)
    scl_water = np.where(scl_clip == 255, 255, scl_clip).astype(np.uint8)

    valid = (cbers_water != 255) & (scl_water != 255)

    # Metrics
    m = compute_metrics(cbers_water, scl_water, valid)
    m["label"] = label
    all_metrics.append(m)

    print(f"  Valid pixels: {valid.sum():,d}")
    print(f"  CBERS-4 water: {(cbers_water[valid]==1).sum():,d}  |  SCL water: {(scl_water[valid]==1).sum():,d}")
    print(f"  OA={m['OA']:.4f}  Kappa={m['Kappa']:.4f}  F1={m['F1']:.4f}  "
          f"Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}")

    pair_data.append({
        "label": label,
        "cbers_water": cbers_water,
        "scl_water": scl_water,
        "valid": valid,
    })

print(f"\n\nProcessed {len(pair_data)} pairs successfully.")

---
## 5. Metrics Summary Table

In [ ]:
if len(all_metrics) > 0:
    df = pd.DataFrame(all_metrics)
    df = df[["label", "OA", "Kappa", "F1", "Precision", "Recall", "TP", "FP", "FN", "TN"]]
    
    print("\nAccuracy Metrics — All Pairs")
    print("=" * 90)
    print(df.to_string(index=False, float_format="{:.4f}".format))
    
    # Averages
    print(f"\nMean across {len(df)} pairs:")
    for col in ["OA", "Kappa", "F1", "Precision", "Recall"]:
        print(f"  {col:12s}: {df[col].mean():.4f} ± {df[col].std():.4f}")
    
    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")
else:
    print("No pairs were processed.")

---
## 6. Side-by-Side Comparison — All Pairs

In [ ]:
n = len(pair_data)
if n == 0:
    print("No pairs to plot.")
else:
    fig, axes = plt.subplots(3, n, figsize=(5 * n, 14))
    if n == 1:
        axes = axes.reshape(3, 1)

    water_cmap = ListedColormap(["#e0e0e0", "#0055cc"])
    water_norm = BoundaryNorm([-0.5, 0.5, 1.5], water_cmap.N)

    agree_cmap = ListedColormap(["#e0e0e0", "#0055cc", "#ff6600", "#00aa44"])
    agree_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], agree_cmap.N)

    for i, pd_entry in enumerate(pair_data):
        label = pd_entry["label"]
        cw = pd_entry["cbers_water"]
        sw = pd_entry["scl_water"]
        v = pd_entry["valid"]

        # Row 0: CBERS-4
        d0 = cw.astype(float); d0[~v] = np.nan
        axes[0, i].imshow(d0, cmap=water_cmap, norm=water_norm, interpolation="none")
        axes[0, i].set_title(f"{label}\nCBERS-4 ({(cw[v]==1).sum():,d} px)", fontsize=9)
        axes[0, i].axis("off")

        # Row 1: Sentinel-2 SCL
        d1 = sw.astype(float); d1[~v] = np.nan
        axes[1, i].imshow(d1, cmap=water_cmap, norm=water_norm, interpolation="none")
        axes[1, i].set_title(f"S2 SCL ({(sw[v]==1).sum():,d} px)", fontsize=9)
        axes[1, i].axis("off")

        # Row 2: Agreement
        c = (cw == 1) & v
        s = (sw == 1) & v
        agree = np.full(cw.shape, np.nan)
        agree[~c & ~s & v] = 0  # TN
        agree[c & s]       = 1  # TP
        agree[c & ~s]      = 2  # FP
        agree[~c & s]      = 3  # FN
        axes[2, i].imshow(agree, cmap=agree_cmap, norm=agree_norm, interpolation="none")
        m = all_metrics[i]
        axes[2, i].set_title(f"Agreement\nF1={m['F1']:.3f}  κ={m['Kappa']:.3f}", fontsize=9)
        axes[2, i].axis("off")

    # Row labels
    axes[0, 0].set_ylabel("CBERS-4", fontsize=11, rotation=0, labelpad=60, va="center")
    axes[1, 0].set_ylabel("S2 SCL", fontsize=11, rotation=0, labelpad=60, va="center")
    axes[2, 0].set_ylabel("Agreement", fontsize=11, rotation=0, labelpad=60, va="center")

    # Legend
    legend_elements = [
        Patch(facecolor="#e0e0e0", edgecolor="gray", label="TN (both non-water)"),
        Patch(facecolor="#0055cc", label="TP (both water)"),
        Patch(facecolor="#ff6600", label="FP (CBERS-4 only)"),
        Patch(facecolor="#00aa44", label="FN (S2 only)"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4,
               fontsize=9, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(f"Multi-Scene Comparison — {TARGET_RES} m resolution", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "01_multi_comparison.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

---
## 7. Metrics Bar Chart

In [ ]:
if len(all_metrics) > 1:
    df = pd.DataFrame(all_metrics)
    metrics_to_plot = ["OA", "Kappa", "F1", "Precision", "Recall"]

    x = np.arange(len(df))
    width = 0.15

    fig, ax = plt.subplots(figsize=(max(8, 3 * len(df)), 5))
    for j, metric in enumerate(metrics_to_plot):
        ax.bar(x + j * width, df[metric], width, label=metric)

    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(df["label"], rotation=45, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.set_title("Accuracy Metrics per Scene Pair")
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "02_metrics_chart.png"), dpi=150)
    plt.show()
elif len(all_metrics) == 1:
    m = all_metrics[0]
    print(f"Single pair metrics for {m['label']}:")
    for k in ["OA", "Kappa", "F1", "Precision", "Recall"]:
        print(f"  {k}: {m[k]:.4f}")

---
## 8. Summary

### Outputs

| File | Contents |
|------|----------|
| `metrics_summary.csv` | OA, Kappa, F1, Precision, Recall for each pair |
| `01_multi_comparison.png` | Side-by-side + agreement maps for all pairs |
| `02_metrics_chart.png` | Bar chart comparing metrics across pairs |

In [ ]:
print("Done.")